# Sheep adapter trimming — cutadapt only

`PRJNA900592` sheep adapter-trim notebook. Source evidence: SMARTer 5′ RACE + NEBNext Ultra II Library Prep for Illumina, MiSeq v3 2×300, 30% PhiX. This stage removes only Illumina/NEBNext read-through adapters plus 3′ quality/length filtering. SMARTer / gene-specific 5′ RACE primers are handled in `primer_trim_sheep.ipynb`, not here.


In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
print("cutadapt", shutil.which("cutadapt"))



In [ ]:
from pathlib import Path

TRIM_QUALITY = '0,30'
MIN_LENGTH = 250
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10

# PRJNA900592 sheep: source evidence = SMARTer 5′ RACE + NEBNext Ultra II Library Prep for Illumina.
# This adapter notebook trims ONLY Illumina/NEBNext read-through adapter contamination plus 3′ quality/length.
# It intentionally does NOT use fastp --detect_adapter_for_pe: fastp detects the abundant SMARTer 5′ RACE primer
# (CTAATACGACTCACTATAGGGCAAGCAGTGGTATCAACGCAGAGT) as an "adapter". That belongs to the sheep technical-primer stage.
ILLUMINA_COMMON_STEM = 'AGATCGGAAGAGC'
ILLUMINA_ADAPTER_R1 = 'AGATCGGAAGAGCACACGTCTGAACTCCAGTCA'
ILLUMINA_ADAPTER_R2 = 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'



In [ ]:
def run_adapter_trim_sheep(volume, dataset='PRJNA900592'):
    vol = Path(volume)
    src = vol / 'raw' / dataset
    base = vol / 'results' / dataset / 'trimmed'
    out = base / 'fastq'

    if base.exists():
        shutil.rmtree(base)
    out.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(f.name.rsplit('_', 1)[0] for f in src.glob('*.fastq.gz')))
    print(f'[adapter_trim_sheep] {dataset}: {len(pairs)} pairs')
    print(f'[adapter_trim_sheep] quality={TRIM_QUALITY} minlen={MIN_LENGTH} times={ADAPTER_TIMES} overlap={ADAPTER_MIN_OVERLAP}')

    for bn in pairs:
        r1 = src / f'{bn}_1.fastq.gz'
        r2 = src / f'{bn}_2.fastq.gz'
        if not r1.exists() or not r2.exists():
            continue
        o1 = out / f'{bn}_1.trim.fastq.gz'
        o2 = out / f'{bn}_2.trim.fastq.gz'

        ca = [
            'cutadapt', '--quality-cutoff', TRIM_QUALITY,
            '--compression-level', '1',
            '-m', str(MIN_LENGTH),
            '--times', str(ADAPTER_TIMES),
            '-O', str(ADAPTER_MIN_OVERLAP),
            # Full Illumina/NEBNext R1/R2 adapters plus the common stem observed by cutadapt probes.
            '-a', ILLUMINA_ADAPTER_R1,
            '-A', ILLUMINA_ADAPTER_R2,
            '-a', ILLUMINA_COMMON_STEM,
            '-A', ILLUMINA_COMMON_STEM,
            '-o', str(o1), '-p', str(o2), str(r1), str(r2),
        ]
        print(f'  [{bn}] cutadapt ...')
        rr = subprocess.run(ca, capture_output=True, text=True)
        if rr.returncode != 0:
            raise RuntimeError(f'cutadapt failed {bn}: {rr.stderr[:1200]}')

    n = len(list(out.glob('*.trim.fastq.gz')))
    print(f'[adapter_trim_sheep] DONE: {n} trimmed files')



### Run

Run sheep adapter trimming after reviewing parameters above.


In [ ]:
run_adapter_trim_sheep('/data/user/epishkin', 'PRJNA900592')
